# Application Data Cleaning

This notebook cleans the main application data. It uses the problems found during EDA to decide what needs to be corrected, kept or removed. It also creates the training and test sets that will be used later.

## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 200)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

raw_path = project_root / "data" / "raw" / "application_train.csv"
output_path = project_root / "data" / "interim" / "application_clean.pkl"
split_folder = project_root / "data" / "modeling" / "splits"
audit_folder = project_root / "reports" / "audits"
training_ids_path = split_folder / "training_ids.csv"
test_ids_path = split_folder / "test_ids.csv"
folds_path = split_folder / "training_fold_assignments.csv"

for folder in [output_path.parent, split_folder, audit_folder]:
    folder.mkdir(parents=True, exist_ok=True)
assert raw_path.exists(), "application_train.csv was not found in data/raw/."
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/application_train.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/application_clean.pkl


## Load and check the original data

In [7]:
application_raw = pd.read_csv(raw_path)
application_clean = application_raw.copy()
original_rows, original_columns = application_raw.shape
original_ids = application_raw["SK_ID_CURR"].copy()
original_target = application_raw["TARGET"].copy()

assert application_raw["SK_ID_CURR"].is_unique
assert application_raw["TARGET"].notna().all()
assert set(application_raw["TARGET"].unique()) == {0, 1}
print("Rows:", original_rows)
print("Columns:", original_columns)
print("Default rate:", round(application_raw["TARGET"].mean() * 100, 3), "%")

Rows: 307511
Columns: 122
Default rate: 8.073 %


## Create training and test sets


The data is divided into two parts. The training set contains 80% of the applicants and will be used to build the models. The test set contains the remaining 20% and will be used only for the final model check.

Both sets are created with almost the same percentage of defaults. This makes the comparison fair.

In [10]:
RANDOM_STATE = 2026
training_ids, test_ids = train_test_split(
    application_raw["SK_ID_CURR"], test_size=0.20, random_state=RANDOM_STATE,
    stratify=application_raw["TARGET"],
)
training_ids = training_ids.sort_values().reset_index(drop=True)
test_ids = test_ids.sort_values().reset_index(drop=True)
training_mask = application_raw["SK_ID_CURR"].isin(training_ids)

training_for_rules = application_raw.loc[training_mask].copy()
split_summary = pd.DataFrame([
    {"partition": "training", "applicants": len(training_ids), "defaults": int(training_for_rules["TARGET"].sum()), "default_rate": training_for_rules["TARGET"].mean()},
    {"partition": "test", "applicants": len(test_ids), "defaults": int(application_raw.loc[~training_mask, "TARGET"].sum()), "default_rate": application_raw.loc[~training_mask, "TARGET"].mean()},
])
print("Training-set rows:", len(training_ids))
print("Test-set rows:", len(test_ids))
split_summary

Training-set rows: 246008
Test-set rows: 61503


,partition,applicants,defaults,default_rate
0,training,246008,19860,0.080729
1,test,61503,4965,0.080728


Both sets ended up with almost the same default rate, so the split worked as expected.


### Create five training folds

The training set is divided into five groups. Each group has almost the same percentage of defaults. Later, the model will be trained several times so that every group gets a turn as the validation group. This gives a more reliable model comparison.

In [13]:
fold_data = training_for_rules[["SK_ID_CURR", "TARGET"]].sort_values("SK_ID_CURR").reset_index(drop=True)
fold_data["FOLD"] = 0
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold_number, (_, validation_index) in enumerate(skf.split(fold_data["SK_ID_CURR"], fold_data["TARGET"]), start=1):
    fold_data.loc[validation_index, "FOLD"] = fold_number
fold_data["FOLD"] = fold_data["FOLD"].astype(int)
fold_summary = fold_data.groupby("FOLD").agg(applicants=("SK_ID_CURR", "size"), defaults=("TARGET", "sum"), default_rate=("TARGET", "mean")).reset_index()
fold_summary

,FOLD,applicants,defaults,default_rate
0,1,49202,3972,0.080728
1,2,49202,3972,0.080728
2,3,49202,3972,0.080728
3,4,49201,3972,0.080730
4,5,49201,3972,0.080730


## Clean text and correct known invalid values

The EDA found that `365243` in `DAYS_EMPLOYED` is not a real number of employment days. It is changed to a missing value. A new column is also created to remember which applicants originally had this value.

The four `XNA` values in `CODE_GENDER` are changed to `Unknown`. The text columns are also checked for extra spaces.

In [16]:
categorical_columns_before = application_clean.select_dtypes(exclude="number").columns.tolist()
text_changes = 0
for column in categorical_columns_before:
    original_text = application_clean[column].copy()
    application_clean[column] = application_clean[column].str.strip()
    text_changes += int((original_text.fillna("<MISSING>") != application_clean[column].fillna("<MISSING>")).sum())

employment_sentinel = application_clean["DAYS_EMPLOYED"].eq(365243)
employment_sentinel_count = int(employment_sentinel.sum())
application_clean["DAYS_EMPLOYED_ANOMALY"] = employment_sentinel.astype("int8")
application_clean.loc[employment_sentinel, "DAYS_EMPLOYED"] = np.nan

gender_xna_count = int(application_clean["CODE_GENDER"].eq("XNA").sum())
application_clean["CODE_GENDER"] = application_clean["CODE_GENDER"].replace("XNA", "Unknown")
print("Whitespace changes:", text_changes)
print("DAYS_EMPLOYED sentinels corrected:", employment_sentinel_count)
print("Invalid gender values changed to Unknown:", gender_xna_count)

Whitespace changes: 0
DAYS_EMPLOYED sentinels corrected: 55374
Invalid gender values changed to Unknown: 4


No extra spaces were found in the text values. No applicant rows were removed by any of this.


## Create external-source summary

The EDA showed that the external-source columns are strongly related to default. However, `EXT_SOURCE_1` has more than 50% missing values.

A new mean feature is created using the external scores that are available for each applicant. Two more columns record how many external scores are available and whether all three scores are missing.

In [19]:
external_source_columns = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application_clean["EXT_SOURCE_MEAN"] = application_clean[external_source_columns].mean(axis=1)
application_clean["EXT_SOURCE_COUNT"] = application_clean[external_source_columns].notna().sum(axis=1).astype("int8")
application_clean["EXT_SOURCE_ALL_MISSING"] = application_clean["EXT_SOURCE_COUNT"].eq(0).astype("int8")
print(application_clean[["EXT_SOURCE_MEAN", "EXT_SOURCE_COUNT", "EXT_SOURCE_ALL_MISSING"]].describe().round(4))

       EXT_SOURCE_MEAN  EXT_SOURCE_COUNT  EXT_SOURCE_ALL_MISSING
count      307339.0000       307511.0000             307511.0000
mean            0.5093            2.2358                  0.0006
std             0.1498            0.6500                  0.0236
min             0.0000            0.0000                  0.0000
25%             0.4136            2.0000                  0.0000
50%             0.5245            2.0000                  0.0000
75%             0.6228            3.0000                  0.0000
max             0.8789            3.0000                  1.0000


Almost every applicant has at least one external score. The new mean feature keeps this information even when one of the original scores is missing.


## Select features using the training set

The feature-removal rules are calculated using only the training set. This keeps the test set separate from the cleaning decisions.

A feature is removed when:

- at least 50% of its values are missing;
- it has only one value; or
- almost every applicant has the same value and its correlation with default is very close to zero.

A feature is not removed only because it has a low correlation. It may still be useful when combined with other features.

In [22]:
MISSING_THRESHOLD = 0.50
NEAR_CONSTANT_THRESHOLD = 0.999
LOW_CORRELATION_THRESHOLD = 0.005
protected_columns = {"CODE_GENDER"}
rule_data = application_clean.loc[application_clean["SK_ID_CURR"].isin(training_ids)].copy()
decision_rows = []

for column in application_clean.columns:
    if column in ["SK_ID_CURR", "TARGET"]:
        continue
    series = rule_data[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    dominant_share = series.value_counts(dropna=False, normalize=True).iloc[0]
    target_association = np.nan
    association_method = "Not calculated"
    if pd.api.types.is_numeric_dtype(series) and unique_non_missing > 1:
        target_association = series.corr(rule_data["TARGET"])
        association_method = "Pearson correlation"

    decision = "Keep"
    reason = "Kept for later feature creation and modelling"
    if column in protected_columns:
        decision = "Keep for fairness audit"
        reason = "Protected attribute is retained for subgroup checking and excluded from model predictors"
    elif missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-set missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set"
    elif pd.notna(target_association) and dominant_share >= NEAR_CONSTANT_THRESHOLD and abs(target_association) < LOW_CORRELATION_THRESHOLD:
        decision = "Remove"
        reason = "Almost every applicant has the same value and its correlation with default is close to zero"

    decision_rows.append({
        "feature": column, "data_type": str(series.dtype), "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "dominant_share": dominant_share, "target_association": target_association,
        "association_method": association_method, "decision": decision, "reason": reason,
    })

feature_decisions = pd.DataFrame(decision_rows).sort_values(["decision", "missing_rate"], ascending=[True, False]).reset_index(drop=True)
feature_decisions["decision"].value_counts()

decision
Keep                       71
Remove                     52
Keep for fairness audit     1
Name: count, dtype: int64

In [23]:
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
application_clean = application_clean.drop(columns=removed_features)
print("Features removed:", len(removed_features))
display(feature_decisions.loc[feature_decisions["decision"] == "Remove", ["feature", "missing_rate", "dominant_share", "target_association", "reason"]].round(6))
print("EXT_SOURCE_1 retained:", "EXT_SOURCE_1" in application_clean.columns)
print("EXT_SOURCE_MEAN retained:", "EXT_SOURCE_MEAN" in application_clean.columns)

Features removed: 52


,feature,missing_rate,dominant_share,target_association,reason
72,COMMONAREA_AVG,0.698518,0.698518,-0.018826,Training-set missing rate is at least 50%
73,COMMONAREA_MODE,0.698518,0.698518,-0.016614,Training-set missing rate is at least 50%
74,COMMONAREA_MEDI,0.698518,0.698518,-0.018864,Training-set missing rate is at least 50%
75,NONLIVINGAPARTMENTS_AVG,0.693912,0.693912,-0.003394,Training-set missing rate is at least 50%
76,NONLIVINGAPARTMENTS_MODE,0.693912,0.693912,-0.001699,Training-set missing rate is at least 50%
77,NONLIVINGAPARTMENTS_MEDI,0.693912,0.693912,-0.002798,Training-set missing rate is at least 50%
78,FONDKAPREMONT_MODE,0.683466,0.683466,NaN,Training-set missing rate is at least 50%
79,LIVINGAPARTMENTS_AVG,0.683246,0.683246,-0.024000,Training-set missing rate is at least 50%
80,LIVINGAPARTMENTS_MODE,0.683246,0.683246,-0.022516,Training-set missing rate is at least 50%
81,LIVINGAPARTMENTS_MEDI,0.683246,0.683246,-0.023556,Training-set missing rate is at least 50%


EXT_SOURCE_1 retained: False
EXT_SOURCE_MEAN retained: True


52 features were removed in total. 41 of them were missing more than half their values, and the other 11 were basically constant with almost no connection to default. Most of the high-missing ones were building or housing columns. EXT_SOURCE_1 was one of them, but its information is already captured in the new external-source mean.


## Handle categorical missing values

Missing text values are changed to `Unknown`. This allows the model to recognize that the original value was not available.

Two new columns are also created. They show the number and percentage of missing values for each applicant.

In [26]:
retained_categorical_columns = application_clean.select_dtypes(exclude="number").columns.tolist()
categorical_missing_before = int(application_clean[retained_categorical_columns].isna().sum().sum())
application_clean[retained_categorical_columns] = application_clean[retained_categorical_columns].fillna("Unknown")
predictor_columns_now = [column for column in application_clean.columns if column not in ["SK_ID_CURR", "TARGET"]]
application_clean["APPLICATION_MISSING_COUNT"] = application_clean[predictor_columns_now].isna().sum(axis=1).astype("int16")
application_clean["APPLICATION_MISSING_RATE"] = application_clean["APPLICATION_MISSING_COUNT"] / len(predictor_columns_now)
print("Categorical missing values filled:", categorical_missing_before)
print("Remaining categorical missing values:", int(application_clean[retained_categorical_columns].isna().sum().sum()))
print(application_clean[["APPLICATION_MISSING_COUNT", "APPLICATION_MISSING_RATE"]].describe().round(4))

Categorical missing values filled: 243438
Remaining categorical missing values: 0
       APPLICATION_MISSING_COUNT  APPLICATION_MISSING_RATE
count                307511.0000               307511.0000
mean                      4.6443                    0.0645
std                       4.2878                    0.0596
min                       0.0000                    0.0000
25%                       0.0000                    0.0000
50%                       7.0000                    0.0972
75%                       7.0000                    0.0972
max                      19.0000                    0.2639


Numerical missing values are left as they are here. They will be handled later during model training, using only the training data, so the test set does not influence that process.


## Summarize the cleaning changes

This table shows how many values or features were affected by each cleaning step. The number of applicant rows stayed the same.

In [29]:
cleaning_audit = pd.DataFrame([
    {"rule": "Whitespace removed from categorical values", "values_affected": text_changes},
    {"rule": "DAYS_EMPLOYED sentinel replaced with missing", "values_affected": employment_sentinel_count},
    {"rule": "DAYS_EMPLOYED_ANOMALY created", "values_affected": employment_sentinel_count},
    {"rule": "CODE_GENDER XNA changed to Unknown", "values_affected": gender_xna_count},
    {"rule": "Categorical missing values changed to Unknown", "values_affected": categorical_missing_before},
    {"rule": "Features removed using training-set rules", "values_affected": len(removed_features)},
])
cleaning_audit

,rule,values_affected
0,Whitespace removed from categorical values,0
1,DAYS_EMPLOYED sentinel replaced with missing,55374
2,DAYS_EMPLOYED_ANOMALY created,55374
3,CODE_GENDER XNA changed to Unknown,4
4,Categorical missing values changed to Unknown,243438
5,Features removed using training-set rules,52


## Check the cleaned dataset

In [31]:
numeric_clean = application_clean.select_dtypes(include="number")
infinite_count = int(np.isinf(numeric_clean.to_numpy()).sum())
retained_rule_features = [column for column in application_clean.columns if column in feature_decisions["feature"].tolist()]
retained_high_missing = [column for column in retained_rule_features if rule_data[column].isna().mean() >= MISSING_THRESHOLD]

validation_checks = pd.DataFrame([
    {"check": "Row count preserved", "passed": len(application_clean) == original_rows},
    {"check": "Applicant IDs unchanged", "passed": application_clean["SK_ID_CURR"].equals(original_ids)},
    {"check": "Applicant IDs remain unique", "passed": application_clean["SK_ID_CURR"].is_unique},
    {"check": "TARGET unchanged", "passed": application_clean["TARGET"].equals(original_target)},
    {"check": "TARGET remains binary", "passed": set(application_clean["TARGET"].unique()) == {0, 1}},
    {"check": "Training and test do not overlap", "passed": len(set(training_ids).intersection(set(test_ids))) == 0},
    {"check": "All rows assigned to training or test set", "passed": len(training_ids) + len(test_ids) == original_rows},
    {"check": "Five training folds created", "passed": fold_data["FOLD"].nunique() == 5},
    {"check": "DAYS_EMPLOYED sentinel removed", "passed": not application_clean["DAYS_EMPLOYED"].eq(365243).any()},
    {"check": "CODE_GENDER XNA removed", "passed": not application_clean["CODE_GENDER"].eq("XNA").any()},
    {"check": "No categorical missing values remain", "passed": application_clean.select_dtypes(exclude="number").isna().sum().sum() == 0},
    {"check": "No retained training feature has at least 50 percent missingness", "passed": len(retained_high_missing) == 0},
    {"check": "EXT_SOURCE_1 removed", "passed": "EXT_SOURCE_1" not in application_clean.columns},
    {"check": "External source summary retained", "passed": {"EXT_SOURCE_MEAN", "EXT_SOURCE_COUNT"}.issubset(application_clean.columns)},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one revised application cleaning check failed."
validation_checks

,check,passed
0,Row count preserved,True
1,Applicant IDs unchanged,True
2,Applicant IDs remain unique,True
3,TARGET unchanged,True
4,TARGET remains binary,True
5,Training and test do not overlap,True
6,All rows assigned to training or test set,True
7,Five training folds created,True
8,DAYS_EMPLOYED sentinel removed,True
9,CODE_GENDER XNA removed,True


All checks passed.


## Save the cleaned data and split information

In [34]:
application_clean.to_pickle(output_path)
pd.DataFrame({"SK_ID_CURR": training_ids}).to_csv(training_ids_path, index=False)
pd.DataFrame({"SK_ID_CURR": test_ids}).to_csv(test_ids_path, index=False)
fold_data[["SK_ID_CURR", "FOLD"]].to_csv(folds_path, index=False)
split_summary.to_csv(audit_folder / "application_split_summary.csv", index=False)
fold_summary.to_csv(audit_folder / "application_fold_summary.csv", index=False)
feature_decisions.to_csv(audit_folder / "application_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "application_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "application_cleaning_validation.csv", index=False)

print("Clean dataset saved:", output_path)
print("Training IDs saved:", training_ids_path)
print("Test IDs saved:", test_ids_path)
print("Fold assignments saved:", folds_path)
print("Output rows:", len(application_clean))
print("Output columns:", application_clean.shape[1])
print("Remaining numerical missing values:", int(application_clean.select_dtypes(include="number").isna().sum().sum()))

Clean dataset saved: /Users/taranveersingh/A-MRP/data/interim/application_clean.pkl
Training IDs saved: /Users/taranveersingh/A-MRP/data/modeling/splits/training_ids.csv
Test IDs saved: /Users/taranveersingh/A-MRP/data/modeling/splits/test_ids.csv
Fold assignments saved: /Users/taranveersingh/A-MRP/data/modeling/splits/training_fold_assignments.csv
Output rows: 307511
Output columns: 76
Remaining numerical missing values: 1428174


## Main cleaning results

The application dataset still contains all 307,511 applicants. No applicant IDs or target values were changed.

The data was divided into 246,008 training applicants and 61,503 test applicants. The training set was also divided into five groups for later model comparison.

The invalid employment value was changed to missing, and a new column was created to record where it appeared. The invalid gender values and missing text values were changed to `Unknown`.

A total of 52 features were removed. Most were removed because at least half of their values were missing. `EXT_SOURCE_1` was removed, but its available information was included in a new external-source mean.

The cleaned dataset has 76 columns. Some numerical missing values remain because they will be handled during model training. The next step is to clean the other data tables.